In [0]:
import mlflow
import mlflow.spark
from pyspark.sql import functions as F
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier, LogisticRegression
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Load features
df_train = spark.table("health.default.provider_fraud_features")

# Handle class imbalance
fraud_count = df_train.filter(F.col("is_fraud") == 1).count()
non_fraud_count = df_train.filter(F.col("is_fraud") == 0).count()
balance_ratio = non_fraud_count / fraud_count

print(f"Fraud cases: {fraud_count}, Non-fraud: {non_fraud_count}, Ratio: {balance_ratio:.2f}")

# Oversample minority class or use class weights
df_fraud = df_train.filter(F.col("is_fraud") == 1)
df_non_fraud = df_train.filter(F.col("is_fraud") == 0).sample(withReplacement=False, fraction=0.3, seed=42)

df_balanced = df_fraud.unionByName(df_non_fraud)

# Feature selection
feature_cols = [
    "total_claims", "unique_patients", "avg_claim_amount", 
    "claims_per_patient", "inpatient_ratio", "reimbursement_zscore",
    "cpp_zscore", "diagnosis_diversity_ratio"
]

# Train-test split
train, test = df_balanced.randomSplit([0.8, 0.2], seed=42)

# MLflow experiment
mlflow.set_experiment("/Users/sanyjain@asu.edu/healthcare_fraud_detection")
model_tmp_dir = "/Volumes/health/default/dataset/mlflow_tmp"

with mlflow.start_run(run_name="random_forest_fraud"):
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("features", feature_cols)
    mlflow.log_param("class_balance_method", "undersampling")
    
    # Pipeline
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
    scaler = StandardScaler(inputCol="features_raw", outputCol="features")
    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol="is_fraud",
        numTrees=200,
        maxDepth=15,
        featureSubsetStrategy="sqrt"
    )
    
    pipeline = Pipeline(stages=[assembler, scaler, rf])
    model = pipeline.fit(train)
    
    # Evaluate
    predictions = model.transform(test)
    
    # Metrics
    auc_evaluator = BinaryClassificationEvaluator(labelCol="is_fraud", metricName="areaUnderROC")
    precision_evaluator = MulticlassClassificationEvaluator(
        labelCol="is_fraud", predictionCol="prediction", metricName="weightedPrecision"
    )
    recall_evaluator = MulticlassClassificationEvaluator(
        labelCol="is_fraud", predictionCol="prediction", metricName="weightedRecall"
    )
    
    auc = auc_evaluator.evaluate(predictions)
    precision = precision_evaluator.evaluate(predictions)
    recall = recall_evaluator.evaluate(predictions)
    
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    
    # Feature importance
    rf_model = model.stages[-1]
    feature_importance = list(zip(feature_cols, rf_model.featureImportances.toArray()))
    feature_importance.sort(key=lambda x: x[1], reverse=True)
    
    print("\nFeature Importance:")
    for feat, importance in feature_importance:
        print(f"  {feat}: {importance:.4f}")
        mlflow.log_metric(f"importance_{feat}", importance)
    
    mlflow.spark.log_model(model, "fraud_model", dfs_tmpdir=model_tmp_dir)
    
    print(f"\nAUC: {auc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")

In [0]:
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.stat import Correlation
import mlflow

with mlflow.start_run(run_name="statistical_anomaly_detection"):
    df_all_providers = spark.table("health.default.gold_provider_fraud_risk")
    
    feature_cols = [
        "total_claims", "unique_patients", "avg_claim_amount", 
        "claims_per_patient", "inpatient_ratio", "reimbursement_zscore",
        "cpp_zscore", "diagnosis_diversity_ratio"
    ]
    
    # Fill nulls
    df_clean = df_all_providers.fillna(0, subset=feature_cols)
    
    # Calculate z-scores for each feature
    for col in feature_cols:
        if col not in ["reimbursement_zscore", "cpp_zscore"]:  # Skip already-normalized
            mean_val = df_clean.agg(F.mean(col)).first()[0]
            std_val = df_clean.agg(F.stddev(col)).first()[0]
            
            df_clean = df_clean.withColumn(
                f"{col}_zscore",
                (F.col(col) - F.lit(mean_val)) / F.lit(std_val if std_val > 0 else 1)
            )
    
    # Calculate composite anomaly score (Euclidean distance in z-score space)
    zscore_cols = [f"{col}_zscore" for col in feature_cols if col not in ["reimbursement_zscore", "cpp_zscore"]]
    zscore_cols.extend(["reimbursement_zscore", "cpp_zscore"])
    
    # Create composite score
    df_clean = df_clean.withColumn(
        "anomaly_score_squared",
        sum([F.pow(F.col(c), 2) for c in zscore_cols])
    ).withColumn(
        "anomaly_score",
        F.sqrt(F.col("anomaly_score_squared"))
    )
    
    # Flag top 5% as anomalies
    threshold = df_clean.approxQuantile("anomaly_score", [0.95], 0.01)[0]
    
    mlflow.log_param("method", "statistical_zscore")
    mlflow.log_param("threshold_percentile", 95)
    mlflow.log_metric("anomaly_threshold", threshold)
    
    df_anomalies = df_clean.withColumn(
        "is_anomaly",
        F.when(F.col("anomaly_score") >= threshold, 1).otherwise(0)
    )
    
    # Join with fraud labels
    df_results = df_anomalies.join(
        spark.table("health.default.bronze_fraud_labels").select(
            "Provider",
            F.col("is_fraud").alias("label_is_fraud")
        ),
        "Provider",
        "left"
    ).fillna(0, subset=["label_is_fraud"])
    
    # Calculate metrics using aggregations (no .toPandas())
    tp = df_results.filter((F.col("is_anomaly") == 1) & (F.col("label_is_fraud") == 1)).count()
    fp = df_results.filter((F.col("is_anomaly") == 1) & (F.col("label_is_fraud") == 0)).count()
    fn = df_results.filter((F.col("is_anomaly") == 0) & (F.col("label_is_fraud") == 1)).count()
    tn = df_results.filter((F.col("is_anomaly") == 0) & (F.col("label_is_fraud") == 0)).count()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    mlflow.log_metric("anomaly_precision", precision)
    mlflow.log_metric("anomaly_recall", recall)
    mlflow.log_metric("anomaly_f1", f1)
    mlflow.log_metric("true_positives", tp)
    mlflow.log_metric("false_positives", fp)
    
    print(f"\nStatistical Anomaly Detection Results:")
    print(f"Threshold (95th percentile): {threshold:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"Confusion Matrix:")
    print(f"  TP: {tp}, FP: {fp}")
    print(f"  FN: {fn}, TN: {tn}")
    
    # Display top anomalies
    top_anomalies = df_results \
        .filter(F.col("is_anomaly") == 1) \
        .select(
            "Provider",
            "anomaly_score",
            "label_is_fraud",
            "total_claims",
            "total_reimbursement",
            "claims_per_patient"
        ) \
        .orderBy(F.desc("anomaly_score")) \
        .limit(20)
    
    print("\nTop 20 Anomalies:")
    top_anomalies.show(20, truncate=False)
    
    # Save results
    df_results.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("health.default.gold_anomaly_detection_results")

In [0]:
# Predict beneficiary annual costs
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

model_tmp_dir = "/Volumes/health/default/dataset/mlflow_tmp"
df_cost_train = spark.table("health.default.beneficiary_features")

cost_features = [
    "age", "Gender", "chronic_condition_count", "inpatient_visits",
    "outpatient_visits", "unique_providers"
]

assembler = VectorAssembler(inputCols=cost_features, outputCol="features")
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="total_costs",
    maxDepth=10,
    maxIter=100
)

pipeline = Pipeline(stages=[assembler, gbt])

train, test = df_cost_train.randomSplit([0.8, 0.2], seed=42)

with mlflow.start_run(run_name="cost_prediction_gbt"):
    model = pipeline.fit(train)
    predictions = model.transform(test)
    
    rmse_eval = RegressionEvaluator(labelCol="total_costs", metricName="rmse")
    mae_eval = RegressionEvaluator(labelCol="total_costs", metricName="mae")
    r2_eval = RegressionEvaluator(labelCol="total_costs", metricName="r2")
    
    rmse = rmse_eval.evaluate(predictions)
    mae = mae_eval.evaluate(predictions)
    r2 = r2_eval.evaluate(predictions)
    
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.spark.log_model(model, "cost_model", dfs_tmpdir=model_tmp_dir)
    
    print(f"Cost Prediction - RMSE: ${rmse:.2f}, MAE: ${mae:.2f}, R²: {r2:.3f}")